In [1]:
import os
import re
import random

import numpy as np
import torch
import pickle
from tqdm.notebook import tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from plotting_betas import *
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from val_test import val_test
from print_results import *

import warnings
warnings.filterwarnings("ignore")

# Use GPU if available (CPU otherwise)
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)
dtype = torch.float32
SEED = 42

def deterministic(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

deterministic(SEED)
# Enable (as much as possible) deterministic operations
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

tslen = 200

cuda:1


## Load Participant IDs

In [2]:
# Load POMA scores and participant IDs (same logic as in fpca.ipynb)
folder_path = "/mnt/sdb/arafat/stroke_riemann/csv_r"
files = [file for file in os.listdir(folder_path)]
files = sorted(files, key=lambda x: int(x.split('_')[0][2:]))

# Participant ID for each row of dataset (same order as files from csv_r)
participant_ids = [re.search(r'ID(\d+)_', f).group(1) for f in files]
print("First 10 participant_ids:", participant_ids[:10])

First 10 participant_ids: ['1', '2', '3', '4', '5', '6', '7', '8', '10', '11']


## Data Loading

In [3]:
def loading():
    with open('unaligned_betas.pkl','rb') as f:
        betas = pickle.load(f)
    return betas

betas = loading()
print(betas.shape)

(155, 32, 3, 200)


In [4]:
K = 32
M = 3
T = tslen
nsamples = 155

# Flatten each sample into a single feature vector: (n_samples, D)
betas_flat = betas.reshape(nsamples, -1)
print(betas_flat.shape)

(155, 19200)


# Nonlinear VAE on Unaligned Skeletons

In [35]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NonlinearVAE(nn.Module):
    """Nonlinear VAE with BatchNorm"""
    def __init__(self, D, R, H=128, dropout=0.1):
        super().__init__()
        self.W1 = nn.Linear(D, H, bias=False)
        self.W2 = nn.Linear(H, R, bias=False)
        self.fc_logvar = nn.Linear(H, R)
        self.dropout = nn.Dropout(dropout)
        
        # BatchNorm layers
        self.bn1 = nn.BatchNorm1d(H)  
        self.bn2 = nn.BatchNorm1d(H)

    def encode(self, x):
        h = self.W1(x)
        h = self.bn1(h)
        h = torch.tanh(h)
        h = self.dropout(h)
        mu = self.W2(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        hrecon = F.linear(z, self.W2.weight.T)  # Input: (batch_size, R), Weights: (R, H)
        hrecon = self.bn2(hrecon)
        x_hat = F.linear(torch.tanh(hrecon), self.W1.weight.T)  # Input: (batch_size, H), Weights: (H, D)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z


def vae_loss(x, x_hat, mu, logvar, beta=1e-4):
    recon = ((x_hat - x)**2).sum(dim=1)
    kl = 0.5 * (logvar.exp() + mu.pow(2) - 1 - logvar).sum(dim=1)
    return (recon + beta * kl).mean(), recon.mean(), kl.mean()


In [36]:
folder_path = "/mnt/sdb/arafat/stroke_riemann/csv_r"
files = [file for file in os.listdir(folder_path)]
files = sorted(files, key=lambda x: int(x.split('_')[0][2:]))
scores_csv = [file.split('_')[1] for file in files]
scores = [file.split('.')[0] for file in scores_csv]
y_poma = np.array(scores).astype(int)
print(y_poma.shape)

(155,)


## Regression Cross Validation

In [37]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error

# VAE (unsupervised) -> latent mu -> KNN to predict POMA
R = 38
kl_beta = 1e-6
n_folds = 30
max_epochs = 1000
n = betas_flat.shape[0]
D = betas_flat.shape[1]
participant_ids = np.asarray(participant_ids)

a_models = {'KNN': KNeighborsRegressor()}
all_results_nested = {name: {'targets': [], 'preds': [], 'subjects': []} for name in a_models.keys()}
all_results_validation = {name: {'targets': [], 'preds': []} for name in a_models.keys()}

for k in tqdm(range(n_folds), total=n_folds, desc='VAE+KNN folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])

    if len(train_idx) == 0 or len(test_idx) == 0 or len(validation_idx) == 0:
        continue

    X_train = betas_flat[train_idx].astype(np.float32)
    X_val = betas_flat[validation_idx].astype(np.float32)
    X_test = betas_flat[test_idx].astype(np.float32)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    y_train_fold = y_poma[train_idx].astype(np.float32)
    y_validation_fold = y_poma[validation_idx].astype(np.float32)
    y_test_fold = y_poma[test_idx].astype(np.float32)

    fold_seed = SEED + k
    deterministic(fold_seed)
    g = torch.Generator().manual_seed(fold_seed)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train)),
        batch_size=32,
        shuffle=True,
        drop_last=True,
        generator=g,
    )

    vae = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    opt = torch.optim.Adam(vae.parameters(), lr=1e-3)

    # Train VAE for a fixed number of epochs (no early stopping)
    for _ in range(max_epochs):
        vae.train()
        for (x_batch,) in train_loader:
            x_batch = x_batch.to(device, dtype=dtype)
            opt.zero_grad(set_to_none=True)
            x_hat, mu, logvar, _ = vae(x_batch)
            loss, _, _ = vae_loss(x_batch, x_hat, mu, logvar, beta=kl_beta)
            loss.backward()
            opt.step()

    # Encode once to latent space (use mean mu)
    vae.eval()
    with torch.no_grad():
        mu_train, _ = vae.encode(torch.from_numpy(X_train).to(device, dtype=dtype))
        mu_val, _ = vae.encode(torch.from_numpy(X_val).to(device, dtype=dtype))
        mu_test, _ = vae.encode(torch.from_numpy(X_test).to(device, dtype=dtype))

    Z_train = mu_train.detach().cpu().numpy()
    Z_validation = mu_val.detach().cpu().numpy()
    Z_test = mu_test.detach().cpu().numpy()

    for name, model_reg in a_models.items():
        m = type(model_reg)(**model_reg.get_params())
        m.fit(Z_train, y_train_fold)

        train_preds = m.predict(Z_train)
        validation_preds = m.predict(Z_validation)
        test_preds = m.predict(Z_test)

        print(
            f"Fold {k+1} {name} train MAE: {mean_absolute_error(y_train_fold, train_preds):.3f}",
            f"validation MAE: {mean_absolute_error(y_validation_fold, validation_preds):.3f}"
        )

        all_results_validation[name]['targets'].extend(y_validation_fold.tolist())
        all_results_validation[name]['preds'].extend(validation_preds.tolist())

        all_results_nested[name]['targets'].extend(y_test_fold.tolist())
        all_results_nested[name]['preds'].extend(test_preds.tolist())
        all_results_nested[name]['subjects'].extend(participant_ids[test_idx].tolist())

results_nested_df = print_results_regression(all_results_validation, all_results_nested, a_models)
results_nested_df


VAE+KNN folds:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 1 KNN train MAE: 1.946 validation MAE: 4.040
Fold 2 KNN train MAE: 1.519 validation MAE: 8.520
Fold 3 KNN train MAE: 1.534 validation MAE: 6.240
Fold 4 KNN train MAE: 1.691 validation MAE: 6.720
Fold 5 KNN train MAE: 2.076 validation MAE: 3.720
Fold 6 KNN train MAE: 1.954 validation MAE: 1.360
Fold 7 KNN train MAE: 2.080 validation MAE: 0.280
Fold 8 KNN train MAE: 2.244 validation MAE: 1.480
Fold 9 KNN train MAE: 2.030 validation MAE: 0.600
Fold 10 KNN train MAE: 2.177 validation MAE: 3.160
Fold 11 KNN train MAE: 2.109 validation MAE: 1.440
Fold 12 KNN train MAE: 1.975 validation MAE: 1.040
Fold 13 KNN train MAE: 2.014 validation MAE: 1.800
Fold 14 KNN train MAE: 1.981 validation MAE: 0.320
Fold 15 KNN train MAE: 2.124 validation MAE: 0.880
Fold 16 KNN train MAE: 1.828 validation MAE: 3.640
Fold 17 KNN train MAE: 1.581 validation MAE: 5.880
Fold 18 KNN train MAE: 1.452 validation MAE: 8.040
Fold 19 KNN train MAE: 1.658 validation MAE: 6.200
Fold 20 KNN train MAE: 2.106 validation 

,MAE,RMSE,R2,Pearson r,Pearson p
KNN,2.72,4.330775,0.385848,0.623733,4.358940e-18


In [38]:
from ci import *

ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci(
    all_results_nested[name]['targets'],
    all_results_nested[name]['preds'],
    all_results_nested[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,MAE,RMSE,R2,Pearson r
mean,2.72,4.330775,0.385848,0.623733
ci,"[2.328, 3.138]","[3.767, 4.829]","[0.257, 0.48]","[0.515, 0.707]"


## LesionLeft Classification (same subject order as regression)

In [39]:
# Create y_lesion from LesionLeft, aligned to same participant_ids order as X
demo_df = pd.read_csv('demo_data.csv')
id_to_lesion = dict(zip(demo_df['s'].astype(int), demo_df['LesionLeft']))
y_lesion = np.array([id_to_lesion[int(pid)] for pid in participant_ids])

print("LesionLeft class distribution:", np.unique(y_lesion, return_counts=True))
print("y_lesion.shape:", y_lesion.shape)

LesionLeft class distribution: (array([0, 1, 2]), array([ 30,  14, 111]))
y_lesion.shape: (155,)


In [40]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# VAE (unsupervised) -> latent mu -> KNN to predict POMA
R = 38
kl_beta = 1e-6
n_folds = 30
max_epochs = 1000
n = betas_flat.shape[0]
D = betas_flat.shape[1]
participant_ids = np.asarray(participant_ids)

a_models = {'KNN': KNeighborsClassifier(),}
all_results_nested = {name: {'targets': [], 'preds': [], 'subjects': []} for name in a_models.keys()}
all_results_validation = {name: {'targets': [], 'preds': []} for name in a_models.keys()}

for k in tqdm(range(n_folds), total=n_folds, desc='VAE+KNN folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids[j] in train_pids])
    test_idx = np.array([j for j in range(n) if participant_ids[j] in test_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids[j] in validation_pids])

    if len(train_idx) == 0 or len(test_idx) == 0 or len(validation_idx) == 0:
        continue

    X_train = betas_flat[train_idx].astype(np.float32)
    X_val = betas_flat[validation_idx].astype(np.float32)
    X_test = betas_flat[test_idx].astype(np.float32)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    y_train_fold = y_lesion[train_idx].astype(int)
    y_validation_fold = y_lesion[validation_idx].astype(int)
    y_test_fold = y_lesion[test_idx].astype(int)

    fold_seed = SEED + k
    deterministic(fold_seed)
    g = torch.Generator().manual_seed(fold_seed)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(X_train)),
        batch_size=32,
        shuffle=True,
        drop_last=False,
        generator=g,
    )

    vae = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    opt = torch.optim.Adam(vae.parameters(), lr=1e-3)

    # Train VAE for a fixed number of epochs (no early stopping)
    for _ in range(max_epochs):
        vae.train()
        for (x_batch,) in train_loader:
            x_batch = x_batch.to(device, dtype=dtype)
            opt.zero_grad(set_to_none=True)
            x_hat, mu, logvar, _ = vae(x_batch)
            loss, _, _ = vae_loss(x_batch, x_hat, mu, logvar, beta=kl_beta)
            loss.backward()
            opt.step()

    # Encode once to latent space (use mean mu)
    vae.eval()
    with torch.no_grad():
        mu_train, _ = vae.encode(torch.from_numpy(X_train).to(device, dtype=dtype))
        mu_val, _ = vae.encode(torch.from_numpy(X_val).to(device, dtype=dtype))
        mu_test, _ = vae.encode(torch.from_numpy(X_test).to(device, dtype=dtype))

    Z_train = mu_train.detach().cpu().numpy()
    Z_validation = mu_val.detach().cpu().numpy()
    Z_test = mu_test.detach().cpu().numpy()

    for name, model_clf in a_models.items():
        m = type(model_clf)(**model_clf.get_params())
        m.fit(Z_train, y_train_fold)

        train_preds = m.predict(Z_train)
        validation_preds = m.predict(Z_validation)
        test_preds = m.predict(Z_test)

        print(
            f"Fold {k+1} {name} train F1 (macro): {f1_score(y_train_fold, train_preds, average='macro'):.2f}",
            f"validation F1 (macro): {f1_score(y_validation_fold, validation_preds, average='macro'):.2f}"
        )

        all_results_validation[name]['targets'].extend(y_validation_fold.tolist())
        all_results_validation[name]['preds'].extend(validation_preds.tolist())

        all_results_nested[name]['targets'].extend(y_test_fold.tolist())
        all_results_nested[name]['preds'].extend(test_preds.tolist())
        all_results_nested[name]['subjects'].extend(participant_ids[test_idx].tolist())

results_nested_df = print_results_clf(all_results_validation, all_results_nested, a_models)
results_nested_df


VAE+KNN folds:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 1 KNN train F1 (macro): 0.75 validation F1 (macro): 0.13
Fold 2 KNN train F1 (macro): 0.62 validation F1 (macro): 0.22
Fold 3 KNN train F1 (macro): 0.69 validation F1 (macro): 0.30
Fold 4 KNN train F1 (macro): 0.73 validation F1 (macro): 0.39
Fold 5 KNN train F1 (macro): 0.68 validation F1 (macro): 0.62
Fold 6 KNN train F1 (macro): 0.72 validation F1 (macro): 1.00
Fold 7 KNN train F1 (macro): 0.70 validation F1 (macro): 1.00
Fold 8 KNN train F1 (macro): 0.75 validation F1 (macro): 1.00
Fold 9 KNN train F1 (macro): 0.72 validation F1 (macro): 1.00
Fold 10 KNN train F1 (macro): 0.68 validation F1 (macro): 0.44
Fold 11 KNN train F1 (macro): 0.68 validation F1 (macro): 0.44
Fold 12 KNN train F1 (macro): 0.70 validation F1 (macro): 1.00
Fold 13 KNN train F1 (macro): 0.77 validation F1 (macro): 1.00
Fold 14 KNN train F1 (macro): 0.69 validation F1 (macro): 1.00
Fold 15 KNN train F1 (macro): 0.70 validation F1 (macro): 1.00
Fold 16 KNN train F1 (macro): 0.71 validation F1 (macro): 0.38
F

,Accuracy,F1 (macro),Precision (macro),Recall (macro)
KNN,0.806452,0.610582,0.679279,0.583741


In [41]:
from ci_class import subject_bootstrap_ci_class

ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci_class(
    all_results_nested[name]['targets'],
    all_results_nested[name]['preds'],
    all_results_nested[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,Accuracy,F1 (weighted),F1 (macro),Precision (weighted),Precision (macro),Recall (weighted),Recall (macro)
mean,0.806452,0.78808,0.610582,0.784664,0.679279,0.806452,0.583741
ci,"[0.76, 0.856]","[0.733, 0.843]","[0.511, 0.694]","[0.726, 0.849]","[0.514, 0.848]","[0.76, 0.856]","[0.506, 0.662]"
